# LightGBM classifier 
using native categorical feature support(pandas "category" dtype, passed via categorical_feature).

In [1]:
import os
import sys
import importlib
import scipy
import sklearn
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Paths / constants

In [2]:
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

sys.path.insert(0, BASE_DIR)

TARGET = "Will_Buy_EV"
ID_COL = "id"

TRAIN_PATH = "./dataset/train_fe.csv"
TEST_PATH = "./dataset/test_fe.csv"

YES_NO_MAP = {"Yes": 1, "No": 0}
RANGE_ANXIETY_MAP = {"Low": 0, "Medium": 1, "High": 2}

MODEL_NAME = "lightgbm"
ARTIFACT_DIR = "./artifacts"
MODEL_DIR = "./models"
SUB_PATH = f"{ARTIFACT_DIR}/test_pred_{MODEL_NAME}.csv"
OOF_PATH = f"{ARTIFACT_DIR}/oof_{MODEL_NAME}.npy"

FOLD_IDS_PATH = f"{ARTIFACT_DIR}/fold_ids.npy"
N_SPLITS = 5
SEED = 42

NUM_COLS = [
    "Age", "Annual_Income_USD", "Daily_Commute_km",
    "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work", "Environmental_Concern_Level",
]
CAT_COLS = [
    "Gender", "City_Type", "Current_Car_Type",
    "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level",
]

LGB_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_samples=30,
    objective="binary",
    metric="auc",
    random_state=SEED,          # was cv.SEED
    n_jobs=-1,
    verbose=-1,
)

In [3]:
def load_raw_data(train_path=TRAIN_PATH, test_path=TEST_PATH):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    if train[TARGET].dtype == object:
        train[TARGET] = train[TARGET].map(YES_NO_MAP).astype(int)
    return train, test


def engineer_features(df):
    df = df.copy()
    df["Home_Charging_bin"] = df["Home_Charging_Possible"].map(YES_NO_MAP)
    df["Subsidy_bin"] = df["Subsidy_Available"].map(YES_NO_MAP)
    df["Range_Anxiety_ord"] = df["Range_Anxiety_Level"].map(RANGE_ANXIETY_MAP)
    df["Income_per_Age"] = df["Annual_Income_USD"] / (df["Age"] + 1)
    df["Commute_per_Car"] = df["Daily_Commute_km"] / (df["Number_of_Cars_Owned"] + 1)
    df["Total_Charging_Stations"] = (
        df["Charging_Stations_Near_Home"] + df["Charging_Stations_Near_Work"]
    )
    df["Charging_Station_Ratio"] = df["Charging_Stations_Near_Home"] / (
        df["Charging_Stations_Near_Work"] + 1
    )
    df["Charging_Access_Score"] = df["Total_Charging_Stations"] * df["Home_Charging_bin"]
    df["Subsidy_x_EnvConcern"] = df["Subsidy_bin"] * df["Environmental_Concern_Level"]
    df["Subsidy_x_HomeCharging"] = df["Subsidy_bin"] * df["Home_Charging_bin"]
    df["EnvConcern_minus_RangeAnxiety"] = (
        df["Environmental_Concern_Level"] - df["Range_Anxiety_ord"]
    )
    df["High_Env_Concern"] = (df["Environmental_Concern_Level"] >= 4).astype(int)
    df["Long_Commute"] = (
        df["Daily_Commute_km"] >= df["Daily_Commute_km"].median()
    ).astype(int)
    df["Age_Group"] = pd.cut(df["Age"], bins=[0, 25, 35, 45, 55, 65, 120], labels=False)
    return df


def get_feature_lists(df):
    engineered_num = [
        "Home_Charging_bin", "Subsidy_bin", "Range_Anxiety_ord",
        "Income_per_Age", "Commute_per_Car", "Total_Charging_Stations",
        "Charging_Station_Ratio", "Charging_Access_Score",
        "Subsidy_x_EnvConcern", "Subsidy_x_HomeCharging",
        "EnvConcern_minus_RangeAnxiety", "High_Env_Concern",
        "Long_Commute", "Age_Group",
    ]
    num_cols = NUM_COLS + [c for c in engineered_num if c in df.columns]
    cat_cols = [c for c in CAT_COLS if c in df.columns]
    return num_cols, cat_cols

In [4]:
def make_folds(y, n_splits=N_SPLITS, seed=SEED):
    y = np.asarray(y)
    fold_ids = np.full(len(y), -1, dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (_, valid_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
        fold_ids[valid_idx] = fold
    return fold_ids


def get_or_create_folds(train_df, target_col=TARGET, n_splits=N_SPLITS,
                        seed=SEED, path=FOLD_IDS_PATH):
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    if os.path.exists(path):
        fold_ids = np.load(path)
        if len(fold_ids) == len(train_df):
            return fold_ids
        print("Existing fold file has wrong length, regenerating folds.")
    fold_ids = make_folds(train_df[target_col].values, n_splits=n_splits, seed=seed)
    np.save(path, fold_ids)
    return fold_ids


def fold_split(train_df, fold_ids, fold):
    train_idx = np.where(fold_ids != fold)[0]
    valid_idx = np.where(fold_ids == fold)[0]
    return train_idx, valid_idx


def summarize_oof(y_true, oof_pred, model_name="model"):
    auc = roc_auc_score(y_true, oof_pred)
    print(f"[{model_name}] OOF ROC-AUC: {auc:.5f}")
    return auc

In [5]:
def main():
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

    train, test = load_raw_data()
    train = engineer_features(train)
    test = engineer_features(test)
    num_cols, cat_cols = get_feature_lists(train)
    feature_cols = num_cols + cat_cols

    for c in cat_cols:
        train[c] = train[c].astype("category")
        test[c] = test[c].astype("category")

    y = train[TARGET].values
    fold_ids = get_or_create_folds(train, target_col=TARGET)

    oof_pred = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    fold_scores = []
    importances = np.zeros(len(feature_cols))

    print("=" * 70)
    print(f"LIGHTGBM ({N_SPLITS}-fold CV)")
    print("=" * 70)

    for fold in range(N_SPLITS):
        train_idx, valid_idx = fold_split(train, fold_ids, fold)

        X_train, y_train = train.loc[train_idx, feature_cols], y[train_idx]
        X_valid, y_valid = train.loc[valid_idx, feature_cols], y[valid_idx]

        model = lgb.LGBMClassifier(**LGB_PARAMS)
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            categorical_feature=cat_cols,
            callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(200)]
        )

        valid_pred = model.predict_proba(X_valid)[:, 1]
        oof_pred[valid_idx] = valid_pred

        fold_auc = roc_auc_score(y_valid, valid_pred)
        fold_scores.append(fold_auc)
        print(f"Fold {fold}: AUC = {fold_auc:.5f} (best_iter={model.best_iteration_})")

        test_pred += model.predict_proba(test[feature_cols])[:, 1] / N_SPLITS
        importances += model.feature_importances_ / N_SPLITS
        model.booster_.save_model(f"{MODEL_DIR}/lightgbm_fold{fold}.txt")

    print(f"\nMean fold AUC: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
    summarize_oof(y, oof_pred, MODEL_NAME)

    imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False)
    print("\nTop feature importances:")
    print(imp_df.head(15).to_string(index=False))

    np.save(OOF_PATH, oof_pred)
    pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred}).to_csv(
        SUB_PATH, index=False
    )
    print(f"\nSaved OOF predictions -> {OOF_PATH}")
    print(f"Saved test predictions -> {SUB_PATH}")

In [6]:
if __name__ == "__main__":
    main()

LIGHTGBM (5-fold CV)
[200]	valid_0's auc: 0.940171
[400]	valid_0's auc: 0.940281
Fold 0: AUC = 0.94032 (best_iter=449)
[200]	valid_0's auc: 0.941219
[400]	valid_0's auc: 0.94128
Fold 1: AUC = 0.94128 (best_iter=407)
[200]	valid_0's auc: 0.942579
[400]	valid_0's auc: 0.942618
Fold 2: AUC = 0.94265 (best_iter=340)
[200]	valid_0's auc: 0.942107
[400]	valid_0's auc: 0.942173
Fold 3: AUC = 0.94218 (best_iter=420)
[200]	valid_0's auc: 0.941363
[400]	valid_0's auc: 0.94147
Fold 4: AUC = 0.94149 (best_iter=412)

Mean fold AUC: 0.94158 (+/- 0.00080)
[lightgbm] OOF ROC-AUC: 0.94157

Top feature importances:
                      feature  importance
            Annual_Income_USD      5070.8
             Daily_Commute_km      3155.0
               Income_per_Age      2769.2
              Commute_per_Car      2341.2
                          Age      2227.2
       Charging_Station_Ratio      1853.4
      Total_Charging_Stations      1122.0
  Charging_Stations_Near_Work      1090.4
        Charging_